<a href="https://colab.research.google.com/github/machancejoy-max/colab-git-demo-JOY/blob/main/Assignment_6_PAAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import time
import os

transform = transforms.Compose([
    transforms.ToTensor(),                      # convert image to tensor
    transforms.Normalize((0.5, 0.5, 0.5),       # normalize each channel
                         (0.5, 0.5, 0.5))
])



train_full = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_data = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

In [20]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

train_size = int(0.8 * len(train_full))
val_size = len(train_full) - train_size

train_data, val_data = random_split(train_full, [train_size, val_size])

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)



In [21]:
class BaselineCNN(nn.Module):
    def __init__(self):
        super(BaselineCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BaselineCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 5

train_acc_list = []
val_acc_list = []
train_loss_list = []
val_loss_list = []


In [18]:
for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    # Validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")


Epoch 1/5 | Train Acc: 0.5659 | Val Acc: 0.6840 | Train Loss: 1.2085 | Val Loss: 0.9027
Epoch 2/5 | Train Acc: 0.7052 | Val Acc: 0.7272 | Train Loss: 0.8377 | Val Loss: 0.7681
Epoch 3/5 | Train Acc: 0.7618 | Val Acc: 0.7565 | Train Loss: 0.6724 | Val Loss: 0.7001
Epoch 4/5 | Train Acc: 0.8077 | Val Acc: 0.7583 | Train Loss: 0.5435 | Val Loss: 0.7212
Epoch 5/5 | Train Acc: 0.8463 | Val Acc: 0.7705 | Train Loss: 0.4331 | Val Loss: 0.7299


In [23]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

test_acc = correct / total
print("Test Accuracy:", test_acc)


Test Accuracy: 0.1


In [24]:
torch.save(model.state_dict(), "baseline_cnn.pth")
size_mb = os.path.getsize("baseline_cnn.pth") / (1024 * 1024)
print("Model Size (MB):", size_mb)


Model Size (MB): 2.3782224655151367


In [25]:
sample = next(iter(test_loader))[0][0].unsqueeze(0).to(device)

start = time.time()
_ = model(sample)
end = time.time()

latency_ms = (end - start) * 1000
print("Inference Latency (ms):", latency_ms)


Inference Latency (ms): 12.812137603759766


In [26]:
model = BaselineCNN().to(device)
model.load_state_dict(torch.load("baseline_cnn.pth"))


<All keys matched successfully>

In [27]:
import torch.nn.utils.prune as prune

pruned_model = BaselineCNN().to(device)
pruned_model.load_state_dict(torch.load("baseline_cnn.pth"))

# Collect all layers to prune
parameters_to_prune = []
for module in pruned_model.modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        parameters_to_prune.append((module, 'weight'))

# Apply 50% global unstructured pruning
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.5
)


In [28]:
for module, _ in parameters_to_prune:
    prune.remove(module, 'weight')


In [29]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(pruned_model.parameters(), lr=0.0005)

num_ft_epochs = 5

for epoch in range(num_ft_epochs):
    pruned_model.train()
    running_loss, correct, total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = pruned_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    print(f"[Pruned FT] Epoch {epoch+1}/{num_ft_epochs} | "
          f"Acc: {correct/total:.4f} | Loss: {running_loss/len(train_loader):.4f}")


[Pruned FT] Epoch 1/5 | Acc: 0.5617 | Loss: 1.2257
[Pruned FT] Epoch 2/5 | Acc: 0.7115 | Loss: 0.8202
[Pruned FT] Epoch 3/5 | Acc: 0.7728 | Loss: 0.6518
[Pruned FT] Epoch 4/5 | Acc: 0.8194 | Loss: 0.5145
[Pruned FT] Epoch 5/5 | Acc: 0.8565 | Loss: 0.4076


In [31]:
import torch
import torch.nn as nn
import os
import time

# Helper function to evaluate model accuracy and loss
def evaluate_model(model, data_loader, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    avg_loss = total_loss / len(data_loader)
    accuracy = correct / total
    return accuracy, avg_loss

# Helper function to measure model inference latency
def measure_latency(model, device, data_loader):
    model.eval()
    sample_input = next(iter(data_loader))[0][0].unsqueeze(0).to(device)

    start = time.time()
    _ = model(sample_input)
    end = time.time()

    latency_ms = (end - start) * 1000
    return latency_ms

# Helper function to get model size in MB
def model_size_mb(filepath):
    return os.path.getsize(filepath) / (1024 * 1024)

torch.save(pruned_model.state_dict(), "pruned_cnn.pth")

pruned_acc, pruned_loss = evaluate_model(pruned_model, test_loader, device)
pruned_latency = measure_latency(pruned_model, device, test_loader)
pruned_size = model_size_mb("pruned_cnn.pth")

print(f"Pruned Model — Acc: {pruned_acc:.4f}, Size: {pruned_size:.2f} MB, Latency: {pruned_latency:.3f} ms")


Pruned Model — Acc: 0.7646, Size: 2.38 MB, Latency: 1.602 ms


In [32]:
quant_model = BaselineCNN().to("cpu")
quant_model.load_state_dict(torch.load("baseline_cnn.pth", map_location="cpu"))


<All keys matched successfully>

In [33]:
quantized_model = torch.quantization.quantize_dynamic(
    quant_model,
    {nn.Linear},        # quantize only Linear layers
    dtype=torch.qint8
)


/tmp/ipykernel_89807/1915274837.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


In [34]:
quant_acc, quant_loss = evaluate_model(quantized_model, test_loader, "cpu")

torch.save(quantized_model.state_dict(), "quantized_cnn.pth")
quant_size = model_size_mb("quantized_cnn.pth")
quant_latency = measure_latency(quantized_model, "cpu", test_loader)

print(f"Quantized Model — Acc: {quant_acc:.4f}, Size: {quant_size:.2f} MB, Latency: {quant_latency:.3f} ms")


Quantized Model — Acc: 0.1000, Size: 0.87 MB, Latency: 1.754 ms


In [36]:
baseline_acc, _ = evaluate_model(model, test_loader, device)
baseline_latency = measure_latency(model, device, test_loader)

print("\n=== MODEL COMPARISON ===")
print(f"Baseline:   Acc={baseline_acc:.4f}, Size={baseline_size:.2f} MB, Latency={baseline_latency:.3f} ms")
print(f"Pruned:     Acc={pruned_acc:.4f}, Size={pruned_size:.2f} MB, Latency={pruned_latency:.3f} ms")
print(f"Quantized:  Acc={quant_acc:.4f}, Size={quant_size:.2f} MB, Latency={quant_latency:.3f} ms")


=== MODEL COMPARISON ===
Baseline:   Acc=0.1000, Size=1.45 MB, Latency=1.619 ms
Pruned:     Acc=0.7646, Size=2.38 MB, Latency=1.602 ms
Quantized:  Acc=0.1000, Size=0.87 MB, Latency=1.754 ms
